# 🧱 Desafio — Pipeline de Dados, Qualidade e LGPD (Supply Chain)

## Cenário

Sua equipe de **Engenharia/Governança de Dados** recebeu um *dump* bruto do sistema de
pedidos de uma empresa de Supply Chain. Antes de qualquer análise ou modelo, a área de
dados precisa **documentar, medir, avaliar a conformidade (LGPD) e limpar** esses dados.

> ⏱️ Trabalho em equipe. Cada parte tem **código base** e células **`# TODO`** para completar.

### Roteiro
1. **Dicionário de Dados** — documentar cada coluna (tipo, descrição, é dado pessoal?).
2. **Catálogo de Métricas** — definir e calcular indicadores de negócio.
3. **Checks de Qualidade** — completude, unicidade e consistência.
4. **LGPD** — classificar a sensibilidade dos dados e propor anonimização.
5. **Limpeza** — tratar duplicatas, valores ausentes e outliers, gerando um dataset confiável.

> ⚠️ Os dados são **sintéticos** e contêm problemas **plantados de propósito**
> (duplicatas, ausentes, outliers, inconsistências e dados pessoais fictícios).

---


## ⚙️ Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import hashlib

pd.set_option('display.max_columns', None)
plt.rcParams['axes.grid'] = True

df = pd.read_csv('supplychain_pipeline.csv')
print('Linhas:', len(df), '| Colunas:', df.shape[1])
df.head()

---
## 📖 Parte 1 — Dicionário de Dados

Um pipeline confiável começa por **documentar o que cada coluna significa**.

### 🎯 O Desafio
Escolham **5 colunas** do dataset e documentem cada uma: descrição, tipo
(numérico / texto / data / categórico) e se é **dado pessoal** (sim/não).
Usem o código base para inspecionar os tipos e exemplos.

In [ ]:
# Inspeção rápida para apoiar o dicionário
resumo = pd.DataFrame({
    'tipo_detectado': df.dtypes.astype(str),
    'n_unicos': df.nunique(),
    'exemplo': [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    'pct_ausente': (df.isna().mean() * 100).round(1),
})
resumo

### ✍️ Preencham o dicionário (escolham 5 colunas)

| Coluna | Tipo | Descrição | Dado pessoal? |
|--------|------|-----------|---------------|
| ... | ... | ... | ... |
| ... | ... | ... | ... |
| ... | ... | ... | ... |
| ... | ... | ... | ... |
| ... | ... | ... | ... |


---
## 📐 Parte 2 — Catálogo de Métricas

Um **catálogo de métricas** padroniza *como* cada indicador é calculado, evitando que cada
pessoa calcule de um jeito.

### 🎯 O Desafio
Definam e calculem **1 métrica** de negócio, deixando claro: **nome**, **fórmula** e **valor**.
Já deixamos o **ticket médio** como exemplo pronto — rodem a célula e, se quiserem, troquem por
outra métrica (taxa de atraso, custo médio de frete, % de cancelamento, receita por região).

In [ ]:
# Métrica de exemplo (já pronta): TICKET MÉDIO = média de valor_total
ticket_medio = df['valor_total'].mean()
print('Ticket médio (R$):', round(ticket_medio, 2))

# (Opcional) TODO — troque ou acrescente outra métrica de sua escolha.
# Dica p/ taxa de atraso: comparar data_entrega_real com data_entrega_prevista
#   (convertam antes com pd.to_datetime(..., errors='coerce'))


---
## 🔍 Parte 3 — Checks de Qualidade

Duas dimensões clássicas:
- **Completude:** faltam valores? (`% de nulos` por coluna)
- **Unicidade:** há duplicatas? a chave primária (`id_pedido`) é única?

### 🎯 O Desafio
1. Calculem a **completude** (% de ausentes por coluna).
2. Verifiquem a **unicidade**: linhas totalmente duplicadas e `id_pedido` repetidos.

In [ ]:
# --- COMPLETUDE ---
# TODO — % de ausentes por coluna (mostre só as que têm ausentes)
# Dicas:
#   df.isna()            -> True onde está vazio
#   df.isna().mean()     -> proporção de nulos por coluna (multiplique por 100)
#   use [coluna > 0] para filtrar apenas as colunas com ausentes


# --- UNICIDADE ---
# TODO — duplicatas e checagem da chave primária
# Dicas:
#   df.duplicated().sum()              -> nº de linhas totalmente duplicadas
#   df['id_pedido'].duplicated().sum() -> nº de ids repetidos
#   df['id_pedido'].is_unique          -> True/False se a PK é única


---
## 🔐 Parte 4 — LGPD: Sensibilidade e Anonimização

A LGPD (Lei Geral de Proteção de Dados) classifica:
- **Dado pessoal:** identifica ou pode identificar uma pessoa (nome, CPF, e-mail, telefone).
- **Dado pessoal sensível:** origem racial, saúde, biometria, opinião política, etc.
- **Quase-identificadores:** sozinhos não identificam, mas combinados sim (cidade + nascimento + UF).

### 🎯 O Desafio
As colunas que precisam de tratamento de privacidade são:
**`cliente_nome`**, **`cliente_cpf`**, **`cliente_email`**, **`cliente_telefone`** e
**`cliente_data_nascimento`** (dados pessoais), além de **`cliente_cidade`** e
**`cliente_uf`** (quase-identificadores).

Deixamos **prontas** as funções que mascaram CPF e telefone. Sua tarefa é **montar o
`df_anon`** aplicando as técnicas de proteção indicadas nos comentários.

In [ ]:
# Funções prontas de mascaramento (NÃO precisam alterar)
def mascara_cpf(cpf):
    """Mantém só os 2 últimos dígitos: 123.456.789-00 -> ***.***.**9-00"""
    s = str(cpf)
    return '***.***.**' + s[-4:] if len(s) >= 4 else '***'

def mascara_telefone(tel):
    """Esconde os 8 dígitos finais: +55 (11) 91234-5678 -> +55 (11) *****-****"""
    import re
    return re.sub(r'\d{4,5}-\d{4}$', '*****-****', str(tel))

# Exemplo de uso das funções prontas:
df_anon = df.copy()
df_anon['cliente_cpf'] = df_anon['cliente_cpf'].apply(mascara_cpf)
df_anon['cliente_telefone'] = df_anon['cliente_telefone'].apply(mascara_telefone)

# Agora complete o restante:
# TODO 1 — MINIMIZAÇÃO: remova a coluna 'cliente_nome' (não é necessária p/ análise)
#   df_anon = df_anon.drop(columns=[...])

# TODO 2 — PSEUDONIMIZAÇÃO: troque o e-mail por um hash irreversível
#   Dica: hashlib.sha256(str(x).encode()).hexdigest()[:16]
#   df_anon['cliente_email'] = df_anon['cliente_email'].apply(lambda x: ...)

# TODO 3 — GENERALIZAÇÃO: troque a data de nascimento por uma faixa etária
#   Dica: calcule a idade e use pd.cut(idade, [0,18,30,45,60,120], labels=[...])

df_anon.head()


---
## 🧹 Parte 5 — Limpeza do Dataset

Agora vamos produzir um **dataset mais confiável** tratando dois problemas:
1. **Remover duplicatas** (linhas idênticas e `id_pedido` repetido).
2. **Tratar valores ausentes** (imputar ou marcar).

### 🎯 O Desafio
Implementem a limpeza e mostrem o **antes × depois** (nº de linhas, duplicatas e ausentes).

In [ ]:
# --- DUPLICATAS ---
# TODO — remover linhas totalmente duplicadas e id_pedido repetido
# Dicas:
#   df.drop_duplicates()
#   df.drop_duplicates(subset='id_pedido', keep='first')

# --- VALORES AUSENTES ---
# TODO — tratar ausentes. Sugestões:
#   valor_frete  -> preencher com a mediana: df['valor_frete'].median()
#   regiao       -> preencher com 'Desconhecida'
#   email/telefone -> preencher com 'não informado' (não se inventa dado pessoal)

# TODO — mostrar o antes x depois (linhas, duplicatas, ausentes)


---
## 🏁 Entregáveis

1. Dicionário de dados completo (Parte 1).
2. Catálogo com ≥ 4 métricas e suas fórmulas (Parte 2).
3. Relatório de qualidade: completude, unicidade e consistência (Parte 3).
4. Classificação LGPD + dataset anonimizado (Parte 4).
5. Dataset limpo + comparação antes × depois (Parte 5).

Bom trabalho! 🚀